# Can we predict malignant findings?
Target: `class` (~2% positive). With such imbalance, accuracy is meaningless (always predicting 0 gives ~98%), so the main metric is **PR-AUC (average precision)**, plus ROC-AUC and precision/recall at a chosen threshold.

Plan: EDA → stratified hold-out test set (untouched until the end) → compare models with repeated stratified CV on the training set (preprocessing inside pipelines to avoid leakage) → choose threshold from out-of-fold predictions → evaluate once on the test set with bootstrap CIs.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, StratifiedKFold, cross_val_predict, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve, roc_curve,
                             confusion_matrix, precision_score, recall_score, f1_score)
SEED = 0
df = pd.read_csv('data.csv')
print(df.shape)
print('missing:', df.isna().sum().sum(), ' duplicate rows:', df.duplicated().sum())
print(df['class'].value_counts())
print(f"prevalence = {df['class'].mean():.3%}")
df.describe().T

## EDA

In [ ]:
X = df.drop(columns='class'); y = df['class']
fig, ax = plt.subplots(2, 3, figsize=(13, 6))
for a, col in zip(ax.ravel(), X.columns):
    lo, hi = X[col].quantile([.005, .995])
    bins = np.linspace(lo, hi, 50)
    a.hist(X.loc[y == 0, col].clip(lo, hi), bins, density=True, alpha=.6, label='benign')
    a.hist(X.loc[y == 1, col].clip(lo, hi), bins, density=True, alpha=.6, label='malignant')
    a.set_title(col)
ax[0, 0].legend(); plt.tight_layout(); plt.show()
print('Univariate ROC-AUC per feature (0.5 = no signal):')
print(pd.Series({k: roc_auc_score(y, X[k]) for k in X}).round(3).to_string())
print('\nUnique values per feature:')
print(X.nunique().to_string())
print('\nCorrelations:')
print(X.corr().round(2))

## Duplicate rows
30% of rows are duplicates, but nearly all of that is a single feature vector repeated thousands of times (almost always benign - likely a default/imputed value). Positives are essentially unique, so train/test leakage is minor; we still check robustness to it at the end.

In [ ]:
g = df.groupby(list(X.columns))['class'].agg(['size', 'mean'])
print(g[g['size'] > 1].rename(columns={'mean': 'frac_malignant'}))
print('unique malignant rows:', df[df['class'] == 1].drop_duplicates().shape[0], 'of', df['class'].sum())
big = g['size'].idxmax()
is_big = (X == pd.Series(big, index=X.columns)).all(axis=1)
print('rows equal to the most repeated vector:', is_big.sum(), ' malignant among them:', y[is_big].sum())

## Split: hold out a stratified test set

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
print(len(Xtr), ytr.sum(), 'positives in train;', len(Xte), yte.sum(), 'positives in test')

## Model comparison (repeated stratified 5-fold CV, train only)
With only a couple hundred positives a single split is noisy, so CV is repeated. Imbalance is handled with class weights rather than resampling (no leakage risk; ranking metrics are unaffected anyway).

In [ ]:
models = {
    'dummy (prior)': DummyClassifier(strategy='prior'),
    'logreg': make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    'logreg balanced': make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced')),
    'random forest': RandomForestClassifier(300, min_samples_leaf=5, class_weight='balanced_subsample', n_jobs=-1, random_state=SEED),
    'hist GB': HistGradientBoostingClassifier(learning_rate=.05, max_iter=300, max_leaf_nodes=8, min_samples_leaf=40,
                                              l2_regularization=1, class_weight='balanced', random_state=SEED),
}
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
rows = {}
for n, m in models.items():
    r = cross_validate(m, Xtr, ytr, cv=cv, scoring={'pr': 'average_precision', 'roc': 'roc_auc'})
    rows[n] = {'PR-AUC': r['test_pr'].mean(), 'PR-AUC sd': r['test_pr'].std(),
               'ROC-AUC': r['test_roc'].mean(), 'ROC-AUC sd': r['test_roc'].std()}
res = pd.DataFrame(rows).T.round(3)
print(f'no-skill PR-AUC = prevalence = {ytr.mean():.3f}')
res

## Threshold selection from out-of-fold predictions (train only)

In [ ]:
best = res.drop('dummy (prior)')['PR-AUC'].idxmax()
print('best model:', best)
model = models[best]
oof = cross_val_predict(model, Xtr, ytr, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method='predict_proba')[:, 1]
p, r, t = precision_recall_curve(ytr, oof)
f1 = 2 * p * r / (p + r + 1e-12)
i = np.argmax(f1[:-1]); thr = t[i]
print(f'OOF PR-AUC={average_precision_score(ytr, oof):.3f}; F1-optimal threshold={thr:.3f} (precision={p[i]:.2f}, recall={r[i]:.2f})')
print('Note: the right threshold depends on the cost of a missed malignancy vs. a false alarm; F1 is only a neutral default.')
plt.plot(r, p); plt.scatter(r[i], p[i], c='r', label='chosen'); plt.axhline(ytr.mean(), ls='--', c='gray', label='no skill')
plt.xlabel('recall'); plt.ylabel('precision'); plt.title('Out-of-fold precision-recall (train)'); plt.legend(); plt.show()

## Final evaluation on the held-out test set (used once)

In [ ]:
model.fit(Xtr, ytr)
s = model.predict_proba(Xte)[:, 1]
pred = (s >= thr).astype(int)
print(f'Test PR-AUC={average_precision_score(yte, s):.3f} (no-skill {yte.mean():.3f})  ROC-AUC={roc_auc_score(yte, s):.3f}')
print(f'At threshold {thr:.3f}: precision={precision_score(yte, pred, zero_division=0):.3f} recall={recall_score(yte, pred):.3f} F1={f1_score(yte, pred):.3f}')
print(pd.DataFrame(confusion_matrix(yte, pred), index=['true 0', 'true 1'], columns=['pred 0', 'pred 1']))
rng = np.random.default_rng(SEED); bs = []
for _ in range(1000):
    k = rng.integers(0, len(yte), len(yte)); yy = yte.values[k]
    if yy.sum() == 0: continue
    bs.append((average_precision_score(yy, s[k]), roc_auc_score(yy, s[k])))
bs = np.array(bs)
print('95%% bootstrap CI  PR-AUC: %.3f-%.3f   ROC-AUC: %.3f-%.3f' % (*np.percentile(bs[:, 0], [2.5, 97.5]), *np.percentile(bs[:, 1], [2.5, 97.5])))
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
fpr, tpr, _ = roc_curve(yte, s)
ax[0].plot(fpr, tpr); ax[0].plot([0, 1], [0, 1], '--', c='gray'); ax[0].set(title='ROC (test)', xlabel='FPR', ylabel='TPR')
pp, rr, _ = precision_recall_curve(yte, s)
ax[1].plot(rr, pp); ax[1].axhline(yte.mean(), ls='--', c='gray'); ax[1].set(title='PR (test)', xlabel='recall', ylabel='precision')
plt.tight_layout(); plt.show()

## Feature importance (permutation, test set)

In [ ]:
from sklearn.inspection import permutation_importance
pi = permutation_importance(model, Xte, yte, scoring='average_precision', n_repeats=20, random_state=SEED)
imp = pd.DataFrame({'mean': pi.importances_mean, 'std': pi.importances_std}, index=X.columns).sort_values('mean')
imp['mean'].plot.barh(xerr=imp['std']); plt.xlabel('drop in PR-AUC'); plt.show()

## Robustness: drop the repeated rows

In [ ]:
keep = ~is_big.loc[Xte.index].values
print(f'Test rows kept: {keep.sum()} ({yte[keep].sum()} malignant)')
print(f'PR-AUC={average_precision_score(yte[keep], s[keep]):.3f}  ROC-AUC={roc_auc_score(yte[keep], s[keep]):.3f}')
print(f'precision={precision_score(yte[keep], pred[keep]):.3f} recall={recall_score(yte[keep], pred[keep]):.3f}')
dd = df.drop_duplicates(); Xd, yd = dd.drop(columns='class'), dd['class']
r = cross_validate(models[best], Xd, yd, cv=cv, scoring={'pr': 'average_precision', 'roc': 'roc_auc'})
print(f'CV on de-duplicated data ({len(dd)} rows, {yd.sum()} malignant): PR-AUC={r["test_pr"].mean():.3f}+-{r["test_pr"].std():.3f}  ROC-AUC={r["test_roc"].mean():.3f}')